# CS4104 Computer Graphics & Virtual Reality
## CASE STUDY CIE3 - Case Study 3: Retail and Remote Commerce

**Student:** Munnaluri Rishikesh  
**USN:** 1RVU23CSE291

### Scenario
An interior-design startup wants a WebXR application that lets users place virtual furniture in their homes using AR/VR. This notebook demonstrates illumination, texture/bump mapping, and window-to-viewport transformation.

## 1. Illumination and Color Models
For a virtual red velvet chair to blend with real-world lighting, combine ambient, diffuse and specular reflection.

`I = Ia*ka + Ip*[kd*max(N.L,0) + ks*max(R.V,0)^n]`

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

def normalize(v):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.clip(n, 1e-9, None)

size=320
x=np.linspace(-1,1,size); y=np.linspace(-1,1,size)
xx,yy=np.meshgrid(x,y); rr2=xx**2+yy**2; mask=rr2<=1
zz=np.zeros_like(xx); zz[mask]=np.sqrt(1-rr2[mask])
normals=normalize(np.dstack((xx,yy,zz)))
light=normalize(np.array([0.55,-0.35,0.76])); view=np.array([0,0,1.0])
ndotl=np.clip(np.sum(normals*light,axis=-1),0,1)
reflect=normalize(2*ndotl[...,None]*normals-light)
rdotv=np.clip(np.sum(reflect*view,axis=-1),0,1)
ambient=0.24; diffuse=0.72*ndotl; specular=0.58*(rdotv**34)
weave=0.035*(np.sin(70*xx)*np.sin(74*yy)); red=np.array([0.68,0.035,0.045])
def make_img(a,d,s):
    out=red[None,None,:]*(a+d)[...,None] + s[...,None]*np.array([1.0,0.82,0.72]) + weave[...,None]
    out=np.clip(out,0,1); out[~mask]=0.95; return out
imgs=[make_img(ambient,np.zeros_like(diffuse),np.zeros_like(specular)),make_img(ambient,diffuse,np.zeros_like(specular)),make_img(ambient,diffuse,specular)]
fig,ax=plt.subplots(1,3,figsize=(11,4))
for a,im,t in zip(ax,imgs,['Ambient only','Ambient + Diffuse','Ambient + Diffuse + Specular']):
    a.imshow(im); a.set_title(t); a.axis('off')
fig.suptitle('Virtual Red Velvet Material - Phong Components')
plt.tight_layout(); plt.show()


## 2. Texture and Bump Mapping
**Texture mapping** adds leather color and grain through `(u,v)` coordinates. **Bump mapping** perturbs surface normals during lighting so detail appears raised or recessed without changing polygon count.

In [ ]:
h,w=280,420
x=np.linspace(0,1,w); y=np.linspace(0,1,h); xx,yy=np.meshgrid(x,y)
grain=0.55*np.sin(95*xx+7*np.sin(12*yy))+0.35*np.sin(128*yy)+0.25*np.sin(170*(xx+yy))
grain/=np.max(np.abs(grain)); seam=np.exp(-((yy-0.5)**2)/0.0007)
albedo=np.clip(0.56+0.075*grain-0.18*seam,0,1)
base=np.dstack([0.30+0.22*albedo,0.13+0.10*albedo,0.055+0.04*albedo])
height=0.58+0.12*grain-0.10*seam; gy,gx=np.gradient(height)
normals=normalize(np.dstack([-2.2*gx,-2.2*gy,np.ones_like(gx)]))
light=normalize(np.array([-0.42,-0.34,0.84])); D=np.clip(np.sum(normals*light,axis=-1),0.10,1)
bumped=np.clip(base*(0.43+0.82*D[...,None]),0,1)
fig,ax=plt.subplots(1,3,figsize=(11,4))
ax[0].imshow(np.ones_like(base)*np.array([0.45,0.22,0.08])); ax[0].set_title('Flat material')
ax[1].imshow(base); ax[1].set_title('Texture mapped')
ax[2].imshow(bumped); ax[2].set_title('Texture + bump mapped')
for a in ax: a.axis('off')
fig.suptitle('Leather Detail Without Extra Polygons'); plt.tight_layout(); plt.show()


## 3. Window-to-Viewport Transformation
`xv = xv_min + (xw - xw_min) * (xv_max - xv_min)/(xw_max - xw_min)`

Use the same form for y so logical AR coordinates map correctly to device pixels.

In [ ]:
def window_to_viewport(xw,yw,window,viewport):
    xw0,xw1,yw0,yw1=window; xv0,xv1,yv0,yv1=viewport
    xv=xv0+(xw-xw0)*(xv1-xv0)/(xw1-xw0)
    yv=yv0+(yw-yw0)*(yv1-yv0)/(yw1-yw0)
    return xv,yv
chair=np.array([[-0.72,-0.42],[0.65,-0.42],[0.65,0.45],[-0.72,0.45],[-0.72,-0.42]])
window=(-1,1,-1,1); viewport=(70,1010,120,1800)
mapped=np.array([window_to_viewport(x,y,window,viewport) for x,y in chair])
fig,ax=plt.subplots(1,2,figsize=(9,5))
ax[0].plot(chair[:,0],chair[:,1],marker='o'); ax[0].set_xlim(-1,1); ax[0].set_ylim(-1,1); ax[0].set_aspect('equal'); ax[0].grid(alpha=.3); ax[0].set_title('AR logical window')
ax[1].plot(mapped[:,0],mapped[:,1],marker='o'); ax[1].set_xlim(0,1080); ax[1].set_ylim(0,1920); ax[1].invert_yaxis(); ax[1].set_aspect('equal'); ax[1].grid(alpha=.3); ax[1].set_title('Phone viewport')
plt.tight_layout(); plt.show()
print('Example:', window_to_viewport(0.25,-0.30,window,viewport))


## Performance Notes for WebXR
- Use low/medium-poly furniture meshes.
- Move fine detail to texture/normal maps.
- Compress and mipmap textures.
- Limit real-time lights and use culling/LOD where possible.

## Conclusion
Illumination models provide believable light response, texture/bump mapping gives inexpensive surface realism, and window-to-viewport transformation provides accurate device interaction.